In [0]:
from config import BRONZE_SCHEMA, SILVER_SCHEMA
from utils import dq_check, dq_check_unique
from pyspark.sql import functions as F

In [0]:
custos_bronze = spark.table(f'{BRONZE_SCHEMA}.custos')
# display(custos_bronze)

In [0]:
custos_sem_duplicados = custos_bronze.drop('ingestion_datetime').dropDuplicates(['id_custo'])
_, passou = dq_check_unique('custos_sem_duplicados', 'id_custo único', custos_sem_duplicados, ['id_custo'])
if not passou:
    raise ValueError('Existem registros diferentes para o mesmo id_custo')
_, passou = dq_check('custos_sem_duplicados', 'id_custo não nulo', custos_sem_duplicados, F.col('id_custo').isNotNull())
if not passou:
    raise ValueError('Existem custos sem id_custo')


In [0]:
# é preciso identificar o porque esses custos são negativos.

display(custos_bronze.where(F.col('custo').rlike(r'-')))

In [0]:
#justificando a remoçao de valores negativos

silver_chamados = spark.table(f'{SILVER_SCHEMA}.chamados')
silver_motivos = spark.table(f'{SILVER_SCHEMA}.base_motivos')

df_cc = custos_bronze.drop('ingestion_datetime').join(silver_chamados.drop('ingestion_datetime'), on = 'id_chamado',how = 'left').join(silver_motivos.drop('ingestion_datetime'), on='id_motivo', how='left')

In [0]:
from pyspark.sql.window import Window
negativos = df_cc.where(F.col('custo').rlike('-'))
colunas = ['categoria','criticidade','nome_motivo','id_cliente','id_atendente']
dataframes = [df_cc,negativos]
for group in colunas :
    for df in dataframes:
        (df.groupBy(group).count().withColumn(
            'proporcao',F.col('count')/ F.sum('count').over(Window.partitionBy())
            )
 .withColumn(
     'percentual',
     F.round(F.col('proporcao')*100,2)
 )
 ).orderBy('proporcao').show(15)
 #como os valores negativos possue uma distribuição bem similar em relaçao aos motivos dos chamados
 #e em relação aos atendentes e clientes. ou seja não existe nenhum atendente cometendo o mesmo erro de maneira desprorcional.
#este erro indica um erro de digitação para a planilha.

In [0]:
# Conversão da coluna custo para decimal
custos_silver = (
    custos_sem_duplicados
    .withColumn('custo', F.regexp_replace(F.trim(F.col('custo')), ',', '.'))
    .withColumn('custo', F.regexp_replace(F.col('custo'), '[^0-9.]', ''))
    .withColumn('custo', F.col('custo').cast('decimal(10, 2)'))
)
# display(custos_silver)

In [0]:
display(custos_silver)

In [0]:
custos_silver.write.format('delta').mode('overwrite').saveAsTable(f'{SILVER_SCHEMA}.custos')